In [ ]:
# ============================================
# BATCH DELETE SPELLS
# ============================================

DRY_RUN = False  # ALWAYS test with True first!
DELAY = 1  # Seconds between deletions
VERBOSE = True  # Detailed logging
CLEAR_SPREADSHEET_IDS = True  # Set to True to also clear DDB column in spreadsheet

# Define a filter function to select which spells to delete
# Examples:
#   - Delete all spells: lambda spell: True
#   - Delete spells with specific IDs: lambda spell: spell['id'] in ['3136976', '3136977']
#   - Delete spells by name pattern: lambda spell: 'old' in spell['name'].lower()
#   - Delete spells with ID > X: lambda spell: int(spell['id']) > 3136000

DELETE_FILTER = lambda spell: True  # Default: delete nothing (safe)

# Example filters (uncomment one to use):
# DELETE_FILTER = lambda spell: spell['id'] in ['3136976', '3136977']  # Delete specific IDs
# DELETE_FILTER = lambda spell: int(spell['id']) > 3136900  # Delete recent spells
# DELETE_FILTER = lambda spell: 'ashen breath' in spell['name'].lower()  # Delete by name

print("=" * 60)
print("⚠️  DANGER: BATCH DELETE SPELLS")
print("=" * 60)
if DRY_RUN:
    print("DRY RUN - No spells will be deleted")
else:
    print("⚠️  LIVE MODE - Spells WILL BE PERMANENTLY DELETED!")
print("=" * 60)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Delay between deletions: {DELAY}s")
print(f"Clear spreadsheet IDs: {'YES' if CLEAR_SPREADSHEET_IDS else 'NO'}")
print()

# Fetch all existing spells from D&D Beyond
print("Fetching all homebrew spells from D&D Beyond...")
try:
    all_spells = ddb_api.get_user_spells()
    print(f"✓ Found {len(all_spells)} existing homebrew spells\n")
except Exception as e:
    print(f"✗ Error fetching spells: {e}")
    print("Cannot proceed with deletion")
    import sys
    sys.exit(1)

# Apply filter to select spells to delete
spells_to_delete = [spell for spell in all_spells if DELETE_FILTER(spell)]

print(f"Filter selected {len(spells_to_delete)} spells for deletion:")
print()
for i, spell in enumerate(spells_to_delete, 1):
    print(f"{i:3d}. {spell['name']:50s} (ID: {spell['id']})")
print()

if len(spells_to_delete) == 0:
    print("✓ No spells selected for deletion")
    print("  Update DELETE_FILTER to select spells")
else:
    # Safety check
    if not DRY_RUN:
        print("⚠️  WARNING: You are about to PERMANENTLY DELETE these spells!")
    
    results = {
        "deleted": 0,
        "errors": 0,
        "details": []
    }
    
    spreadsheet_clears = []
    
    for i, spell in enumerate(spells_to_delete, 1):
        spell_name = spell['name']
        spell_id = spell['id']
        
        print(f"[{i}/{len(spells_to_delete)}] Processing: {spell_name} (ID: {spell_id})")
        
        if DRY_RUN:
            print(f"  → DRY RUN: Would delete spell")
            print(f"    URL: {DDB_BASE_URL}/homebrew/creations/spells/{spell_id}")
            results["deleted"] += 1
            results["details"].append({
                "name": spell_name,
                "id": spell_id,
                "action": "dry_run_delete",
                "success": True
            })
        else:
            try:
                if VERBOSE:
                    print(f"  → Deleting spell from D&D Beyond...")
                
                deleted = ddb_api.delete_spell(spell_id, spell_name)
                
                if deleted:
                    print(f"  ✓ Deleted successfully")
                    results["deleted"] += 1
                    results["details"].append({
                        "name": spell_name,
                        "id": spell_id,
                        "action": "deleted",
                        "success": True
                    })
                    
                    # Track for spreadsheet clearing
                    if CLEAR_SPREADSHEET_IDS:
                        spreadsheet_clears.append({
                            'match_value': spell_name,
                            'update_column': 'DDB',
                            'update_value': ''  # Clear the cell
                        })
                else:
                    print(f"  ✗ Failed to delete")
                    if ddb_api.last_error:
                        print(f"    Error: {ddb_api.last_error}")
                    results["errors"] += 1
                    results["details"].append({
                        "name": spell_name,
                        "id": spell_id,
                        "action": "error",
                        "success": False,
                        "error": ddb_api.last_error
                    })
                
                # Rate limiting
                if i < len(spells_to_delete):
                    time.sleep(DELAY)
                    
            except Exception as e:
                print(f"  ✗ Unexpected error: {e}")
                if VERBOSE:
                    import traceback
                    print(f"    Traceback:\n{traceback.format_exc()}")
                results["errors"] += 1
                results["details"].append({
                    "name": spell_name,
                    "id": spell_id,
                    "action": "error",
                    "success": False,
                    "error": str(e)
                })
    
    # Clear DDB IDs from spreadsheet if requested
    if spreadsheet_clears and not DRY_RUN and CLEAR_SPREADSHEET_IDS:
        print("\n" + "=" * 60)
        print("Clearing DDB IDs from Google Sheets...")
        print("=" * 60)
        try:
            update_results = fantasy_sheets.batch_update_cells_by_row_match(
                SPELLS_GID,
                "Spell Name",
                spreadsheet_clears
            )
            success_count = sum(1 for v in update_results.values() if v)
            print(f"✓ Cleared {success_count}/{len(spreadsheet_clears)} DDB IDs from spreadsheet")
        except Exception as e:
            print(f"✗ Error clearing spreadsheet IDs: {e}")
            if VERBOSE:
                import traceback
                print(f"  Traceback:\n{traceback.format_exc()}")
    
    print()
    print("=" * 60)
    print("DELETE COMPLETE")
    print("=" * 60)
    if DRY_RUN:
        print(f"DRY RUN: Would delete {results['deleted']} spells")
        print("\nSet DRY_RUN = False to actually delete")
    else:
        print(f"✓ Deleted: {results['deleted']}")
        print(f"✗ Errors: {results['errors']}")
    print("=" * 60)
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    # Save log
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = f"DNDBeyond/delete_log_{timestamp}.json"
    with open(log_file, 'w') as f:
        json.dump({
            "timestamp": timestamp,
            "dry_run": DRY_RUN,
            "verbose": VERBOSE,
            "clear_spreadsheet_ids": CLEAR_SPREADSHEET_IDS,
            "total": len(spells_to_delete),
            "deleted": results["deleted"],
            "errors": results["errors"],
            "details": results["details"]
        }, f, indent=2)
    
    print(f"\n✓ Log saved to: {log_file}")

# D&D Beyond Homebrew Sync

Sync homebrew spells from Google Sheets to D&D Beyond.

## Setup Instructions

1. **Get your D&D Beyond cookies**:
   - Open D&D Beyond in your browser
   - Log in to your account
   - Open Developer Tools (F12)
   - Go to Network tab
   - Visit any D&D Beyond page
   - Find the request, right-click → Copy → Copy as cURL
   - Extract the Cookie header value

2. **Get security tokens**:
   - Go to https://www.dndbeyond.com/homebrew/creations/create-spell/create
   - Open Developer Tools → Network tab
   - Fill out the form (don't submit yet)
   - Right-click → Inspect on the form
   - Find the hidden `security-token` and `authenticity-token` fields

3. **Configure below** with your cookies and tokens

## Important Notes

- **Security tokens expire**: You'll need to refresh them periodically
- **Rate limiting**: D&D Beyond may throttle requests
- **Master subscription**: Some features require a Master-tier subscription


<input id="field-security-token" name="security-token" type="hidden" value="bf877cec19a71926736b4fc2826c3d9f">

## 1. Setup and Imports

In [1]:
import pandas as pd
import requests
import json
import time
import re
from typing import Dict, List, Optional
from datetime import datetime
from urllib.parse import urlencode
from bs4 import BeautifulSoup

# Import our Google Sheets client
import src.content.fantasy_spells

from DNDBeyond.helpers.DnDBeyondAPI import DnDBeyondAPI

print("✓ Imports loaded")

✓ Imports loaded


## 2. D&D Beyond Authentication

Configure your D&D Beyond cookies and security tokens.

**Note**: The sync will automatically track DDB spell IDs in your Google Sheets by adding them to a "DDB" column.

In [2]:
# ============================================
# D&D Beyond Configuration
# ============================================

# Base URL
DDB_BASE_URL = "https://www.dndbeyond.com"

# Your D&D Beyond cookies (from browser)
# IMPORTANT: Keep these secret! Don't commit to version control!
DDB_COOKIES = '_pxhd=PQ/qoKAoCogxxx1-vsTXz/ULpK5j8JUBz9hRrkWCYiib5KEoXdqrG0TfrC1sW3ZsNyfvempb-QNw9eGp7MMvmA==:3tq0Zk/-MxlMgdCz9QRMUf6uKjwwhlbq0oruLmZuIxJfpVW8FQYPuYGXUWZYvZ6HdwcW5ZTKM2INFRRiyLZg7kPZZKRy7/TUhLPzrXtbLfw=; ResponsiveSwitch.DesktopMode=1; _swb=181c66c9-d81b-499d-bd23-b8630529b73a; _ga=GA1.2.299146768.1739101198; LoginState=7c207547-4516-49e3-b73b-3b6173e7ddbd; Geo={%22region%22:%22QC%22%2C%22country%22:%22CA%22%2C%22continent%22:%22NA%22}; ddb_vid=b0c44884-c9f1-45bc-bce2-64a98c848c63; _gcl_au=1.1.235369019.1766500827; AWSALB=oLfNrjFnpmS7XlfaW2PKCn3bVFuxYK5bmaVDQ2umex3IDpxH2KH+XD/LTatKY7YihBBN/d33qz3opEtwVvIfzdtHFD4hyWEyQ4UiyS6dg5+5nwgKvjfznj+4dgYK; AWSALBCORS=oLfNrjFnpmS7XlfaW2PKCn3bVFuxYK5bmaVDQ2umex3IDpxH2KH+XD/LTatKY7YihBBN/d33qz3opEtwVvIfzdtHFD4hyWEyQ4UiyS6dg5+5nwgKvjfznj+4dgYK; g_state={"i_l":0,"i_ll":1766500891465}; _gsid=a03bf0d6a9a3456bb54d079e730bae3e; LoginStateWizards=M3xmZmRlMmY0NTA0Mzc0ZWNlYmNlMjA2ZTE3Y2I1NmJjMHwvY2hhcmFjdGVycw; Platform=ddb.web; CobaltSession=eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..cEIaVPBQQBMWcXO6h_TutQ.we_PzUm05tVDfSt54DtGmxsNwFR4SC2AzTXoZo8-uy5neWS_O3zsfW1KpteACO9H.L85HjtaOjs_OUtWjd3dKEA; User.ID=110746825; User.Username=Rockoteque; Preferences.Language=1; UserInfo={"UserId":110746825,"UserJoinDate":"2021-07-13","UserSessionId":"667ae0ed-b6fa-4094-b255-ff429bab2bfe"}; Preferences.TimeZoneID=1; Preferences=undefined; sublevel=MASTER; ddb.toast.spell.homebrew-create.hide-toast=true; ddb.toast.spell.homebrew-edit.hide-toast=true; RequestVerificationToken=acfa0630-890d-493a-8572-a12bf2933551; ddbSiteBanner:d3eb8c76-93aa-4a89-aaff-795f1d14b52e=true; ddb.toast.race.homebrew-create.hide-toast=true; ddb.toast.race.homebrew-edit.hide-toast=true'

# Security tokens (need to be refreshed from the create form)
DDB_SECURITY_TOKEN = "180ee79a4d11a492157db66c75490e68"
DDB_AUTHENTICITY_TOKEN = "eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..hpOtYABR0Iljk7B9-hm02g.wMBIwO_gbXEMWzQYywehGxhGIA621Ri-mJpafeKbe4rG6-63ZLNulZ3ArFbQZoWf.wwyoQE-wk8xgyrrnPPFaog"

REQUEST_VERIFICATION_TOKEN = "acfa0630-890d-493a-8572-a12bf2933551"

# User info
DDB_USER_ID = "110746825"  # Your user ID
DDB_USERNAME = "Rockoteque"  # Your username

# ============================================
# Setup Session
# ============================================

session = requests.Session()

if DDB_COOKIES:
    session.headers.update({
        "Cookie": DDB_COOKIES,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br, zstd",
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:147.0) Gecko/20100101 Firefox/147.0",
        "Referer": f"{DDB_BASE_URL}/homebrew/creations/create-spell/create",
        "Origin": DDB_BASE_URL,
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "same-origin",
        "Sec-Fetch-User": "?1",
        "Upgrade-Insecure-Requests": "1"
    })
    print("✓ Using cookie authentication")
    print(f"✓ Base URL: {DDB_BASE_URL}")
    print(f"✓ User: {DDB_USERNAME} (ID: {DDB_USER_ID})")
    
    if DDB_SECURITY_TOKEN and DDB_AUTHENTICITY_TOKEN:
        print("✓ Security tokens configured")
    else:
        print("⚠️  Security tokens not set - you'll need these to create spells!")
else:
    print("✗ ERROR: No cookies provided!")
    print("   Please set DDB_COOKIES with your browser session cookies")

✓ Using cookie authentication
✓ Base URL: https://www.dndbeyond.com
✓ User: Rockoteque (ID: 110746825)
✓ Security tokens configured


## 3. Load Spell Data from Google Sheets

In [3]:
# Load fantasy spells
print("Loading spells from Google Sheets...")
spells = src.content.fantasy_spells.spells_list

print(f"✓ Loaded {len(spells)} spells")
print(f"\nSample spell:")
if spells:
    sample = spells[0]
    print(f"  Name: {sample.get('name', 'N/A')}")
    print(f"  Level: {sample.get('level', 'N/A')}")
    print(f"  School: {sample.get('school', 'N/A')}")

# Also load the raw DataFrame to check for existing DDB IDs
print("\nLoading spreadsheet data for DDB ID tracking...")
SPELLS_GID = "625265890"  # Fantasy spells sheet GID
from src.content.gsheets_client import fantasy_sheets

df_spells = fantasy_sheets.get_sheet(SPELLS_GID)
print(f"✓ Loaded spreadsheet with {len(df_spells)} rows")

# Ensure DDB column exists
print("\nEnsuring 'DDB' column exists in spreadsheet...")
try:
    ddb_col_idx = fantasy_sheets.ensure_column_exists(SPELLS_GID, "DDB")
    print(f"✓ 'DDB' column exists at index {ddb_col_idx}")
    
    # Check if any spells already have DDB IDs
    df_spells = fantasy_sheets.get_sheet(SPELLS_GID)  # Reload after ensuring column
    if 'DDB' in df_spells.columns:
        existing_ddb_ids = df_spells[df_spells['DDB'].notna()]['DDB'].tolist()
        print(f"✓ Found {len(existing_ddb_ids)} spells already synced to D&D Beyond")
    else:
        print("  'DDB' column created")
except Exception as e:
    print(f"⚠️  Could not ensure DDB column: {e}")
    print("  You may need to check your Google Sheets credentials")
    print("  Continuing without spreadsheet tracking...")

Loading spells from Google Sheets...
✓ Loaded 397 spells

Sample spell:
  Name: Imperial Veil of the Imperatum
  Level: 6
  School: A

Loading spreadsheet data for DDB ID tracking...
✓ Loaded spreadsheet with 401 rows

Ensuring 'DDB' column exists in spreadsheet...
✓ 'DDB' column exists at index 71
✓ Found 0 spells already synced to D&D Beyond


## 4. D&D Beyond API Helper

Helper class for interacting with D&D Beyond's homebrew creation endpoints.

In [4]:


ddb_api = DnDBeyondAPI(session, DDB_SECURITY_TOKEN, DDB_AUTHENTICITY_TOKEN, REQUEST_VERIFICATION_TOKEN)
print("✓ D&D Beyond API helper initialized")

✓ D&D Beyond API helper initialized


In [5]:
from DNDBeyond.helpers import convert_spell_to_ddb

print("✓ D&D Beyond converter loaded")

✓ D&D Beyond converter loaded


## 5.5 Duplicate Checking (HTML Parsing)

The `get_user_spells()` method now uses HTML parsing to fetch existing homebrew spells.

**How it works:**

D&D Beyond uses Server-Side Rendering (SSR) and embeds spell data in the HTML response rather than providing a separate API endpoint. The method:

1. Fetches `/my-creations?filter-type=1118725998` (spell type filter)
2. Parses the HTML to find all elements with `class="list-row-homebrew-creation-Spell"`
3. Extracts spell IDs and names from the `data-slug` attributes
   - Format: `data-slug="3135831-fingerbang"`
   - Parsed to: `{"id": "3135831", "name": "Fingerbang"}`

**Filter Status Options:**
- `filter-status=1` - Published spells only
- `filter-status=2` - Draft spells only
- Use both parameters to fetch all spells

You can test the duplicate checking in the next cell.

In [6]:
# Test fetching existing spells with HTML parsing
print("Testing get_user_spells() method...\n")

try:
    existing_spells = ddb_api.get_user_spells()
    
    if existing_spells:
        print(f"✓ Successfully parsed {len(existing_spells)} existing homebrew spells\n")
        print("Found spells:")
        for i, spell in enumerate(existing_spells, 1):
            spell_name = spell.get('name', 'Unknown')
            spell_id = spell.get('id', 'Unknown')
            print(f"{i:2d}. {spell_name} (ID: {spell_id})")
            print(f"    URL: {DDB_BASE_URL}/homebrew/creations/spells/{spell_id}")
        
        print(f"\n✓ Duplicate checking is ready to use!")
        print(f"  When syncing, spells with matching names will be skipped.")
    else:
        print("⚠️  No spells found")
        print("\nPossible reasons:")
        print("1. You don't have any published homebrew spells yet")
        print("2. Your session cookies may have expired")
        print("3. The HTML structure may have changed")
        print("\nTo include draft spells, update the params in get_user_spells():")
        print('  params = {"filter-type": "1118725998", "filter-status": "2"}')

except Exception as e:
    print(f"✗ Error fetching spells: {e}")
    print("\nThis could mean:")
    print("1. Session cookies have expired - update DDB_COOKIES")
    print("2. D&D Beyond's HTML structure changed")
    print("3. Network or authentication issue")
    print("\nYou can still sync without duplicate checking.")
    import traceback
    traceback.print_exc()

Testing get_user_spells() method...

⚠️  No spells found

Possible reasons:
1. You don't have any published homebrew spells yet
2. Your session cookies may have expired
3. The HTML structure may have changed

To include draft spells, update the params in get_user_spells():
  params = {"filter-type": "1118725998", "filter-status": "2"}


## 6. Sync Spells to D&D Beyond

**Configuration Options:**
- `DRY_RUN`: Set to `False` to actually create/update spells (default: `False`)
- `BATCH_SIZE`: Set to `None` to sync all spells, or a number to limit (e.g., `10` for testing)
- `DELAY`: Seconds between requests to avoid rate limiting (default: `2`)
- `VERBOSE`: Detailed logging for debugging (default: `True`)
- `SKIP_EXISTING`: Set to `True` to skip spells that already exist (default: `True`)
  - When `True`: Spells with DDB IDs in spreadsheet or found on D&D Beyond will be skipped (if `UPDATE_EXISTING=False`)
  - When `False`: Will attempt to create spells even if they already exist (may create duplicates or fail)
- `UPDATE_EXISTING`: **NEW!** Set to `True` to UPDATE spells when DDB ID exists (default: `True`)
  - When `True` and `SKIP_EXISTING=False`: Spells with DDB IDs will be UPDATED instead of skipped
  - This is useful when you rename spells in Google Sheets - the sync will update the spell name on D&D Beyond
  - When `False` or `SKIP_EXISTING=True`: Spells with DDB IDs will be skipped (old behavior)
- `UPDATE_ADDITIONAL_FIELDS`: Set to `True` to update AOE, save, attack fields after creation (default: `True`)

**Typical Usage Scenarios:**

1. **Initial sync (create all spells)**:
   - `SKIP_EXISTING=True`, `UPDATE_EXISTING=False`
   - Skips spells that already exist (based on DDB ID in spreadsheet)

2. **Update renamed spells**:
   - `SKIP_EXISTING=False`, `UPDATE_EXISTING=True`
   - Updates all spells that have DDB IDs (handles name changes)
   - Creates new spells that don't have DDB IDs

3. **Dry run to test**:
   - `DRY_RUN=True`
   - Shows what would happen without making changes

**Error Logging:**
When errors occur, the sync will log:
- HTTP status codes and response details
- Exception types and full tracebacks
- Converted spell data for debugging
- All details are saved to `sync_log_YYYYMMDD_HHMMSS.json`

**⚠️ Important**: 
- D&D Beyond doesn't have a bulk API - this creates/updates spells one by one
- Rate limiting: D&D Beyond may throttle requests (adjust `DELAY` if needed)
- Security tokens expire: You may need to refresh them mid-sync
- Check the spreadsheet after sync - DDB IDs will be automatically recorded

In [7]:
# ============================================
# SYNC SPELLS (WITH UPDATE SUPPORT)
# ============================================

# Import normalize_ddb_id helper to handle pandas float conversion
from DNDBeyond.helpers import normalize_ddb_id

DRY_RUN = False  # Set to False to actually create/update spells
BATCH_SIZE = None  # Set to None for all spells, or a number to limit (e.g., 10)
DELAY = 1  # Seconds between requests
VERBOSE = True  # Set to True for detailed logging
SKIP_EXISTING = True  # Set to False to attempt creating spells even if they already exist  
UPDATE_EXISTING = True  # Set to True to UPDATE spells when DDB ID exists (handles renames)
UPDATE_ADDITIONAL_FIELDS = True  # Set to True to update AOE, save, attack fields after creation

print("=" * 60)
if DRY_RUN:
    print("DRY RUN - No spells will be created/updated")
else:
    print("⚠️  LIVE MODE - Spells WILL be created/updated!")
print("=" * 60)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Batch size: {'ALL SPELLS' if BATCH_SIZE is None else BATCH_SIZE}")
print(f"Delay between requests: {DELAY}s")
print(f"Verbose logging: {'ON' if VERBOSE else 'OFF'}")
print(f"Skip existing: {'YES' if SKIP_EXISTING else 'NO (will attempt to create anyway)'}")
print(f"Update existing: {'YES (will update when DDB ID exists)' if UPDATE_EXISTING else 'NO'}")
print(f"Update additional fields: {'YES' if UPDATE_ADDITIONAL_FIELDS else 'NO'}")
print()

# Select spells to sync
spells_to_sync = spells if BATCH_SIZE is None else spells[:BATCH_SIZE]
print(f"Total spells to process: {len(spells_to_sync)}")
print()

# Check spreadsheet for existing DDB IDs first
print("Checking spreadsheet for existing DDB IDs...")
df_spells = fantasy_sheets.get_sheet(SPELLS_GID)
spell_to_ddb_id = {}
if 'DDB' in df_spells.columns and 'Spell Name' in df_spells.columns:
    for _, row in df_spells.iterrows():
        spell_name = row.get('Spell Name')
        ddb_id = row.get('DDB')
        if spell_name and pd.notna(ddb_id):
            normalized_id = normalize_ddb_id(ddb_id)
            if normalized_id:
                spell_to_ddb_id[spell_name] = normalized_id
    print(f"✓ Found {len(spell_to_ddb_id)} spells with DDB IDs in spreadsheet")
else:
    print("  No DDB IDs found in spreadsheet")

# Fetch existing spells from D&D Beyond HTML (backup check)
print("\nFetching existing homebrew spells from D&D Beyond...")
if not DRY_RUN:
    try:
        existing_spells = ddb_api.get_user_spells()
        print(f"✓ Found {len(existing_spells)} existing homebrew spells\n")
    except Exception as e:
        print(f"⚠️  Could not fetch existing spells: {e}")
        print("Using spreadsheet DDB IDs only...\n")
        existing_spells = []
else:
    existing_spells = []

results = {
    "created": 0,
    "updated": 0,
    "skipped": 0,
    "errors": 0,
    "details": []
}

# Track updates to write back to spreadsheet
spreadsheet_updates = []

for i, spell in enumerate(spells_to_sync, 1):
    spell_name = spell.get("name", "Unnamed")
    print(f"[{i}/{len(spells_to_sync)}] Processing: {spell_name}")
    
    try:
        # Convert to D&D Beyond format first (needed for both create and update)
        if VERBOSE:
            print(f"  → Converting spell to D&D Beyond format...")
        
        try:
            ddb_data = convert_spell_to_ddb(spell)
            if VERBOSE:
                print(f"    Level: {ddb_data['level']}, School ID: {ddb_data['school_id']}")
                print(f"    Components: V={ddb_data['verbal']}, S={ddb_data['somatic']}, M={ddb_data['material']}")
                print(f"    Range: {ddb_data['range']}, Classes: {len(ddb_data.get('classes', []))}")
                if ddb_data.get('aoe_type'):
                    print(f"    AOE: Type {ddb_data['aoe_type']}, Size {ddb_data['aoe_size']}")
                if ddb_data.get('save_type'):
                    print(f"    Save: Type {ddb_data['save_type']}")
                if ddb_data.get('attack_type'):
                    print(f"    Attack: Type {ddb_data['attack_type']}")
        except Exception as conv_error:
            print(f"  ✗ Conversion failed: {conv_error}")
            if VERBOSE:
                import traceback
                print(f"    Traceback:\n{traceback.format_exc()}")
            results["errors"] += 1
            results["details"].append({
                "title": spell_name,
                "action": "error",
                "success": False,
                "error": f"Conversion error: {str(conv_error)}",
                "error_type": "conversion",
                "spell_data": {k: str(v)[:100] for k, v in spell.items()}
            })
            continue
        
        # Check if spell already has DDB ID in spreadsheet
        existing_id = spell_to_ddb_id.get(spell_name)
        
        if existing_id:
            # Spell has a DDB ID - decide whether to UPDATE or SKIP
            if UPDATE_EXISTING and not SKIP_EXISTING:
                # UPDATE mode: Update the existing spell
                if DRY_RUN:
                    print(f"  → DRY RUN: Would update spell (ID: {existing_id})")
                    results["details"].append({
                        "title": spell_name,
                        "action": "dry_run_update",
                        "success": True,
                        "id": existing_id
                    })
                    results["updated"] += 1
                else:
                    print(f"  → Updating existing spell (ID: {existing_id})")
                    if VERBOSE:
                        print(f"    URL: {DDB_BASE_URL}/homebrew/creations/spells/{existing_id}")
                    
                    slug = DnDBeyondAPI.create_slug(spell_name)
                    updated = ddb_api.update_basic_information(existing_id, slug, ddb_data)
                    
                    if updated:
                        print(f"  ✓ Updated spell successfully")
                        results["updated"] += 1
                        results["details"].append({
                            "title": spell_name,
                            "action": "updated",
                            "success": True,
                            "id": existing_id
                        })
                    else:
                        print(f"  ✗ Failed to update spell")
                        error_info = {
                            "title": spell_name,
                            "action": "error",
                            "success": False,
                            "error_type": "update_failed",
                            "id": existing_id
                        }
                        if ddb_api.last_error:
                            error_info["error"] = ddb_api.last_error
                        results["errors"] += 1
                        results["details"].append(error_info)
                    
                    # Rate limiting
                    if i < len(spells_to_sync):
                        time.sleep(DELAY)
            else:
                # SKIP mode: Just skip the spell
                print(f"  ✓ Already synced (DDB ID: {existing_id} from spreadsheet)")
                if VERBOSE:
                    print(f"    URL: {DDB_BASE_URL}/homebrew/creations/spells/{existing_id}")
                results["skipped"] += 1
                results["details"].append({
                    "title": spell_name,
                    "action": "skipped",
                    "success": True,
                    "id": existing_id,
                    "reason": "already_in_spreadsheet"
                })
            continue
        
        # No DDB ID in spreadsheet - check D&D Beyond directly (backup check)
        if SKIP_EXISTING and not DRY_RUN and existing_spells:
            existing_id = ddb_api.find_spell_by_name(spell_name, existing_spells)
            if existing_id:
                print(f"  ⚠️  Spell exists on D&D Beyond (ID: {existing_id})")
                if VERBOSE:
                    print(f"    URL: {DDB_BASE_URL}/homebrew/creations/spells/{existing_id}")
                print(f"    Updating spreadsheet with ID...")
                
                spreadsheet_updates.append({
                    'match_value': spell_name,
                    'update_column': 'DDB',
                    'update_value': existing_id
                })
                
                results["skipped"] += 1
                results["details"].append({
                    "title": spell_name,
                    "action": "skipped",
                    "success": True,
                    "id": existing_id,
                    "reason": "already_exists_ddb"
                })
                continue
        
        # No existing ID found - CREATE new spell
        if DRY_RUN:
            print(f"  → DRY RUN: Would create spell")
            print(f"    Level: {ddb_data['level']}, School: {ddb_data['school_id']}")
            results["details"].append({
                "title": spell_name,
                "action": "dry_run_create",
                "success": True
            })
            results["created"] += 1
        else:
            # Create spell
            if VERBOSE:
                print(f"  → Creating spell on D&D Beyond...")
            
            spell_id = ddb_api.create_spell(ddb_data)
            
            if spell_id:
                print(f"  ✓ Created: ID {spell_id}")
                print(f"    URL: {DDB_BASE_URL}/homebrew/creations/spells/{spell_id}")
                
                # Update additional fields (AOE, save, attack) if enabled
                if UPDATE_ADDITIONAL_FIELDS:
                    if VERBOSE:
                        print(f"  → Updating additional fields (AOE, save, attack)...")
                    
                    # Create slug for the update endpoint
                    slug = DnDBeyondAPI.create_slug(spell_name)
                    
                    updated = ddb_api.update_basic_information(spell_id, slug, ddb_data)
                    
                    if updated:
                        if VERBOSE:
                            details = []
                            if ddb_data.get('aoe_type'):
                                details.append(f"AOE:{ddb_data['aoe_type']}")
                            if ddb_data.get('save_type'):
                                details.append(f"Save:{ddb_data['save_type']}")
                            if ddb_data.get('attack_type'):
                                details.append(f"Attack:{ddb_data['attack_type']}")
                            detail_str = ", ".join(details) if details else "none detected"
                            print(f"    ✓ Additional fields updated ({detail_str})")
                    else:
                        print(f"    ⚠️  Warning: Could not update additional fields")
                
                if VERBOSE:
                    print(f"    Adding to spreadsheet update queue...")
                
                # Add to batch update
                spreadsheet_updates.append({
                    'match_value': spell_name,
                    'update_column': 'DDB',
                    'update_value': spell_id
                })
                
                results["created"] += 1
                results["details"].append({
                    "title": spell_name,
                    "action": "created",
                    "success": True,
                    "id": spell_id
                })
                
                # Add to existing list for subsequent checks
                existing_spells.append({"name": spell_name, "id": spell_id})
            else:
                print(f"  ✗ Failed to create spell")
                
                # Log detailed error information
                error_info = {
                    "title": spell_name,
                    "action": "error",
                    "success": False,
                    "error_type": "creation_failed"
                }
                
                # Include last error details
                if ddb_api.last_error:
                    if isinstance(ddb_api.last_error, dict):
                        error_info["error"] = ddb_api.last_error
                        if "status_code" in ddb_api.last_error:
                            print(f"    HTTP {ddb_api.last_error['status_code']}: {ddb_api.last_error.get('reason', 'Unknown')}")
                        elif "exception_type" in ddb_api.last_error:
                            print(f"    Exception: {ddb_api.last_error['exception_type']}")
                    else:
                        error_info["error"] = str(ddb_api.last_error)
                        print(f"    Error: {ddb_api.last_error}")
                else:
                    error_info["error"] = "Unknown error - no details available"
                    print(f"    Error: Unknown (no details)")
                
                # Include converted data for debugging
                if VERBOSE:
                    error_info["converted_data"] = {
                        k: str(v)[:200] if isinstance(v, str) else v 
                        for k, v in ddb_data.items()
                    }
                
                results["errors"] += 1
                results["details"].append(error_info)
            
            # Rate limiting
            if i < len(spells_to_sync):
                time.sleep(DELAY)
    
    except Exception as e:
        print(f"  ✗ Unexpected error: {e}")
        if VERBOSE:
            import traceback
            print(f"    Traceback:\n{traceback.format_exc()}")
        
        results["errors"] += 1
        results["details"].append({
            "title": spell_name,
            "action": "error",
            "success": False,
            "error": str(e),
            "error_type": "unexpected_exception",
            "traceback": traceback.format_exc() if VERBOSE else None
        })

# Write DDB IDs back to spreadsheet
if spreadsheet_updates and not DRY_RUN:
    print("\n" + "=" * 60)
    print("Updating Google Sheets with DDB IDs...")
    print("=" * 60)
    try:
        update_results = fantasy_sheets.batch_update_cells_by_row_match(
            SPELLS_GID,
            "Spell Name",
            spreadsheet_updates
        )
        success_count = sum(1 for v in update_results.values() if v)
        print(f"✓ Updated {success_count}/{len(spreadsheet_updates)} spells in spreadsheet")
        
        # Log any failures
        failed = [k for k, v in update_results.items() if not v]
        if failed:
            print(f"⚠️  Failed to update {len(failed)} spells:")
            for spell_name in failed[:5]:
                print(f"  - {spell_name}")
            if len(failed) > 5:
                print(f"  ... and {len(failed) - 5} more")
    except Exception as e:
        print(f"✗ Error updating spreadsheet: {e}")
        print("  Spell IDs were created but not recorded in spreadsheet")
        if VERBOSE:
            import traceback
            print(f"  Traceback:\n{traceback.format_exc()}")

print()
print("=" * 60)
print("SYNC COMPLETE")
print("=" * 60)
if DRY_RUN:
    print(f"DRY RUN: Would create {results['created']} spells")
    print(f"DRY RUN: Would update {results['updated']} spells")
    print("\nSet DRY_RUN = False to actually create/update")
else:
    print(f"✓ Created: {results['created']}")
    print(f"✓ Updated: {results['updated']}")
    if UPDATE_ADDITIONAL_FIELDS:
        print(f"  (Additional fields update count included in 'Updated' above)")
    print(f"⚠️  Skipped (already exist): {results['skipped']}")
    print(f"✗ Errors: {results['errors']}")
    
    if results['errors'] > 0:
        print(f"\nℹ️  Check the log file for detailed error information")
print("=" * 60)
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Save log with enhanced error details
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = f"DNDBeyond/sync_log_{timestamp}.json"
with open(log_file, 'w') as f:
    json.dump({
        "timestamp": timestamp,
        "dry_run": DRY_RUN,
        "verbose": VERBOSE,
        "batch_size": BATCH_SIZE,
        "skip_existing": SKIP_EXISTING,
        "update_existing": UPDATE_EXISTING,
        "update_additional_fields": UPDATE_ADDITIONAL_FIELDS,
        "total": len(spells_to_sync),
        "created": results["created"],
        "updated": results["updated"],
        "skipped": results["skipped"],
        "errors": results["errors"],
        "details": results["details"]
    }, f, indent=2)

print(f"\n✓ Log saved to: {log_file}")

# Print error summary if there were errors
if results["errors"] > 0 and not VERBOSE:
    print(f"\n⚠️  {results['errors']} errors occurred. Set VERBOSE = True for detailed output.")
    print(f"   Or check the log file for full details: {log_file}")

⚠️  LIVE MODE - Spells WILL be created/updated!
Start time: 2025-12-24 11:35:18
Batch size: ALL SPELLS
Delay between requests: 1s
Verbose logging: ON
Skip existing: YES
Update existing: YES (will update when DDB ID exists)
Update additional fields: YES

Total spells to process: 397

Checking spreadsheet for existing DDB IDs...
✓ Found 0 spells with DDB IDs in spreadsheet

Fetching existing homebrew spells from D&D Beyond...
✓ Found 0 existing homebrew spells

[1/397] Processing: Imperial Veil of the Imperatum
  → Converting spell to D&D Beyond format...
    Level: 6, School ID: 3
    Components: V=True, S=True, M=False
    Range: 30, Classes: 2
    AOE: Type 5, Size 30
  → Creating spell on D&D Beyond...
  ✓ Created: ID 3137182
    URL: https://www.dndbeyond.com/homebrew/creations/spells/3137182
  → Updating additional fields (AOE, save, attack)...
    ✓ Additional fields updated (AOE:5)
    Adding to spreadsheet update queue...
[2/397] Processing: Edict of Imperial Suspension
  → Conv

## 7. Batch Delete Spells from D&D Beyond

**⚠️ DANGER ZONE - This will permanently delete spells!**

Use this section to bulk delete spells from D&D Beyond. This is useful for:
- Cleaning up duplicate spells created by mistake
- Removing old versions after renaming spells
- Starting fresh with a clean slate

**How it works:**
1. Fetches all your existing spells from D&D Beyond
2. Allows you to filter spells to delete (by name pattern, ID range, etc.)
3. Deletes the selected spells one by one
4. Optionally clears DDB IDs from the spreadsheet

**Configuration:**
- `DRY_RUN`: Set to `False` to actually delete (always test with `True` first!)
- `DELETE_FILTER`: Function to filter which spells to delete
- `CLEAR_SPREADSHEET_IDS`: Set to `True` to also remove DDB IDs from spreadsheet